# End-to-End ML Training Pipeline with MLflow Tracking

**Dataset:** UCI Heart Disease Dataset  
**Link:** https://archive.ics.uci.edu/dataset/45/heart+disease  
**Alternative (Kaggle):** https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

## Pipeline Overview
1. Data Ingestion from CSV
2. Feature Engineering & Preprocessing
3. Model Training (Random Forest, SVM, Logistic Regression)
4. MLflow Experiment Tracking
5. Model Registry & Versioning
6. Model Drift Detection

In [ ]:
# Install dependencies
!pip install mlflow scikit-learn pandas numpy matplotlib seaborn kaggle apache-airflow

In [ ]:
# ── STEP 1: DATA INGESTION ──
import pandas as pd
import numpy as np

# Option A: Download from Kaggle
# !kaggle datasets download -d johnsmith88/heart-disease-dataset --unzip

# Option B: Load directly from UCI URL
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
columns = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(url, names=columns, na_values='?')

print(f'Dataset shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')
df.head()

In [ ]:
# ── STEP 2: FEATURE ENGINEERING & PREPROCESSING ──
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Binarize target (0 = no disease, 1 = disease)
df['target'] = (df['target'] > 0).astype(int)

# Impute missing values
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

X = df_imputed.drop('target', axis=1)
y = df_imputed['target']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

In [ ]:
# ── STEP 3: MLFLOW EXPERIMENT TRACKING ──
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# Start MLflow server locally: mlflow ui --port 5000
mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('heart-disease-pipeline')

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':          SVC(probability=True, random_state=42),
    'LogisticReg':  LogisticRegression(max_iter=1000, random_state=42)
}

best_model, best_f1, best_run_id = None, 0, None

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        f1  = f1_score(y_test, preds)
        auc = roc_auc_score(y_test, probs)
        acc = accuracy_score(y_test, preds)

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({'f1': f1, 'auc': auc, 'accuracy': acc})
        mlflow.sklearn.log_model(model, artifact_path='model')

        print(f'{name} → F1: {f1:.3f} | AUC: {auc:.3f} | Acc: {acc:.3f}')

        if f1 > best_f1:
            best_f1, best_model, best_run_id = f1, model, mlflow.active_run().info.run_id

print(f'\nBest model run_id: {best_run_id}')

In [ ]:
# ── STEP 4: MODEL REGISTRY & VERSIONING ──
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_uri = f'runs:/{best_run_id}/model'

# Register model
reg = mlflow.register_model(model_uri=model_uri, name='HeartDiseaseClassifier')
print(f'Registered model version: {reg.version}')

# Transition to Production
client.transition_model_version_stage(
    name='HeartDiseaseClassifier',
    version=reg.version,
    stage='Production'
)
print('Model transitioned to Production stage')

In [ ]:
# ── STEP 5: MODEL DRIFT DETECTION ──
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings('ignore')

# Simulate live prediction data (replace with real incoming data)
live_data = X_test + np.random.normal(0, 0.3, X_test.shape)

drift_detected = False
drift_report = {}

for i, col in enumerate(X.columns):
    stat, p_value = ks_2samp(X_train[:, i], live_data[:, i])
    drifted = p_value < 0.05
    drift_report[col] = {'ks_stat': round(stat, 4), 'p_value': round(p_value, 4), 'drift': drifted}
    if drifted:
        drift_detected = True

print('── Drift Report ──')
for col, result in drift_report.items():
    status = '⚠ DRIFT' if result['drift'] else '✓ OK'
    print(f'{col:15s} | KS: {result["ks_stat"]} | p: {result["p_value"]} | {status}')

if drift_detected:
    print('\n🚨 ALERT: Drift detected — retraining recommended!')
else:
    print('\n✅ No significant drift detected.')